In [ ]:
"""
=============================================================================
 Customer Churn Prediction - Step 1: Data Loading & Initial Inspection
=============================================================================
 Dataset : IBM Telco Customer Churn
 File    : WA_Fn-UseC_-Telco-Customer-Churn.csv
 Author  : Data Science Team
=============================================================================
"""

import pandas as pd

# -- 1. Load the dataset --------------------------------------------------
FILE_PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(FILE_PATH)

print("=" * 65)
print("  CUSTOMER CHURN PREDICTION - DATA INSPECTION REPORT")
print("=" * 65)

# -- 2. Dataset Shape -----------------------------------------------------
print(f"\n[SHAPE]  Dataset Shape")
print(f"    Rows    : {df.shape[0]:,}")
print(f"    Columns : {df.shape[1]}")

# -- 3. Column Data Types --------------------------------------------------
print(f"\n[TYPES]  Data Types per Column")
print("-" * 40)
for col in df.columns:
    print(f"    {col:<22s} : {str(df[col].dtype)}")

print(f"\n    Summary: {df.dtypes.value_counts().to_dict()}")

# -- 4. First 5 Rows (Preview) --------------------------------------------
print(f"\n[PREVIEW]  First 5 Rows")
print("-" * 65)
print(df.head().to_string(index=False))

# -- 5. Missing / Null Value Analysis --------------------------------------
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_report = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct.round(2)
})

print(f"\n[MISSING]  Missing / Null Values")
print("-" * 45)

if missing.sum() == 0:
    print("    [OK]  No null values detected in any column!")
else:
    # Show only columns with missing values
    cols_with_missing = missing_report[missing_report["Missing Count"] > 0]
    print(cols_with_missing.to_string())

print(f"\n    Total null cells : {missing.sum():,}")
print(f"    Total cells      : {df.shape[0] * df.shape[1]:,}")

# -- 6. Bonus: Check for whitespace-only / empty-string values -------------
#    (Common pitfall in this dataset - TotalCharges has " " entries)
print(f"\n[BLANK]  Whitespace / Empty-String Check (non-null but blank)")
print("-" * 45)
blank_counts = {}
for col in [c for c in df.columns if str(df[c].dtype) in ["object", "string", "str"]]:
    blanks = (df[col].str.strip() == "").sum()
    if blanks > 0:
        blank_counts[col] = blanks

if blank_counts:
    for col, count in blank_counts.items():
        print(f"    {col:<22s} : {count} blank entries")
else:
    print("    [OK]  No blank-string entries found.")

# -- 7. Concise Info -------------------------------------------------------
print(f"\n[INFO]  df.info()")
print("-" * 65)
df.info()

print("\n" + "=" * 65)
print("  END OF INSPECTION REPORT")
print("=" * 65)


In [ ]:
"""
=============================================================================
 Customer Churn Prediction - Step 2: Feature Encoding & Scaling
=============================================================================
 Dataset : IBM Telco Customer Churn
 Goal    : Clean TotalCharges, encode categoricals, then scale numerical
           features (tenure, MonthlyCharges, TotalCharges) for model-ready data
=============================================================================
"""

import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

# =========================================================================
# 1. LOAD DATA
# =========================================================================
FILE_PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(FILE_PATH)

print("=" * 70)
print("  STEP 2 : FEATURE ENCODING & SCALING")
print("=" * 70)

# =========================================================================
# 2. CLEAN TotalCharges (11 blank-string entries --> NaN --> fill with 0)
#    These correspond to brand-new customers with tenure = 0
# =========================================================================
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

blanks_found = df["TotalCharges"].isnull().sum()
print(f"\n[CLEAN]  TotalCharges")
print(f"    Blank entries converted to NaN : {blanks_found}")

# Fill NaN with 0 (tenure=0 customers have no charges yet)
df["TotalCharges"] = df["TotalCharges"].fillna(0)
print(f"    NaN filled with 0              : Done")
print(f"    TotalCharges dtype now          : {df['TotalCharges'].dtype}")

# =========================================================================
# 3. DROP customerID (not a predictive feature)
# =========================================================================
df.drop(columns=["customerID"], inplace=True)
print(f"\n[DROP]   Removed 'customerID' column (non-predictive)")

# =========================================================================
# 4. ENCODE CATEGORICAL FEATURES (Label Encoding)
# =========================================================================
print(f"\n[ENCODE] Label Encoding categorical columns")
print("-" * 50)

label_encoders = {}
categorical_cols = [col for col in df.columns if str(df[col].dtype) in ["object", "string", "str"]]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"    {col:<22s} : {list(le.classes_)}")

print(f"\n    Total columns encoded : {len(categorical_cols)}")
print(f"    Encoded mapping stored in 'label_encoders' dict")

# =========================================================================
# 5. SCALE NUMERICAL FEATURES
# =========================================================================
# Features to scale -- these have very different ranges:
#   tenure         :   0 - 72     (months)
#   MonthlyCharges : ~18 - 118    (dollars)
#   TotalCharges   :   0 - 8700+  (dollars)
#
# StandardScaler transforms each feature to mean=0, std=1
# This is preferred for Logistic Regression and KNN because:
#   - LR: gradient descent converges faster with standardized features
#   - KNN: distance-based, so unscaled features with large ranges dominate

numerical_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

print(f"\n[SCALE]  Scaling numerical features with StandardScaler")
print("-" * 50)

# -- Before scaling: show original statistics --
print(f"\n    BEFORE scaling:")
print(df[numerical_cols].describe().round(2).to_string())

# -- Apply StandardScaler --
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

# -- After scaling: show transformed statistics --
print(f"\n    AFTER scaling (mean ~ 0, std ~ 1):")
print(df[numerical_cols].describe().round(4).to_string())

# -- Print scaler parameters for reference --
print(f"\n    Scaler parameters (for inverse transform / new data):")
for i, col in enumerate(numerical_cols):
    print(f"      {col:<18s} : mean = {scaler.mean_[i]:.4f},  std = {scaler.scale_[i]:.4f}")

# =========================================================================
# 6. FINAL DATASET SUMMARY
# =========================================================================
print(f"\n{'=' * 70}")
print(f"  FINAL DATASET SUMMARY")
print(f"{'=' * 70}")
print(f"    Shape       : {df.shape}")
print(f"    Dtypes      : {df.dtypes.value_counts().to_dict()}")
print(f"    Null values : {df.isnull().sum().sum()}")

print(f"\n    First 5 rows of scaled features:")
print(df[numerical_cols].head().to_string())

print(f"\n    Full DataFrame head:")
print(df.head().to_string())

print(f"\n{'=' * 70}")
print(f"  READY FOR MODEL TRAINING")
print(f"  - Logistic Regression (sklearn.linear_model.LogisticRegression)")
print(f"  - K-Nearest Neighbors (sklearn.neighbors.KNeighborsClassifier)")
print(f"{'=' * 70}")


In [ ]:
"""
=============================================================================
 Customer Churn Prediction - Step 3: Model Training & Evaluation
=============================================================================
 Dataset : IBM Telco Customer Churn (cleaned, encoded, scaled)
 Models  : Logistic Regression & K-Nearest Neighbors (KNN)
 Split   : 80% Training / 20% Testing
=============================================================================
"""

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# =========================================================================
# 1. DATA PREPARATION (from Step 2)
# =========================================================================
FILE_PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(FILE_PATH)

# Clean TotalCharges
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Drop customerID
df.drop(columns=["customerID"], inplace=True)

# Label Encode categoricals
categorical_cols = [col for col in df.columns if str(df[col].dtype) in ["object", "string", "str"]]
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# Scale numerical features
numerical_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

print("=" * 70)
print("  STEP 3 : MODEL TRAINING & EVALUATION")
print("=" * 70)

# =========================================================================
# 2. DEFINE FEATURES (X) AND TARGET (y)
# =========================================================================
X = df.drop(columns=["Churn"])
y = df["Churn"]

print(f"\n[DATA]   Feature matrix shape : {X.shape}")
print(f"         Target vector shape  : {y.shape}")
print(f"         Churn distribution   : No={y.value_counts()[0]}, Yes={y.value_counts()[1]}")
print(f"         Churn rate           : {y.mean():.2%}")

# =========================================================================
# 3. TRAIN-TEST SPLIT (80/20)
# =========================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,       # reproducibility
    stratify=y             # preserve churn ratio in both sets
)

print(f"\n[SPLIT]  80/20 Stratified Split (random_state=42)")
print(f"         Training set : {X_train.shape[0]:,} samples")
print(f"         Testing set  : {X_test.shape[0]:,} samples")
print(f"         Train churn% : {y_train.mean():.2%}")
print(f"         Test churn%  : {y_test.mean():.2%}")

# =========================================================================
# 4. MODEL TRAINING
# =========================================================================

# -- 4a. Logistic Regression ----------------------------------------------
print(f"\n{'=' * 70}")
print(f"  MODEL 1 : LOGISTIC REGRESSION")
print(f"{'=' * 70}")

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    solver="lbfgs"
)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print(f"    Training complete.")

# -- 4b. K-Nearest Neighbors ---------------------------------------------
print(f"\n{'=' * 70}")
print(f"  MODEL 2 : K-NEAREST NEIGHBORS (K=5)")
print(f"{'=' * 70}")

knn_model = KNeighborsClassifier(
    n_neighbors=5,
    metric="minkowski",
    p=2                    # Euclidean distance
)
knn_model.fit(X_train, y_train)
knn_pred = knn_model.predict(X_test)

print(f"    Training complete.")

# =========================================================================
# 5. EVALUATION
# =========================================================================

def evaluate_model(name, y_true, y_pred):
    """Print a full evaluation report for a model."""
    acc   = accuracy_score(y_true, y_pred)
    prec  = precision_score(y_true, y_pred)
    rec   = recall_score(y_true, y_pred)
    f1    = f1_score(y_true, y_pred)
    cm    = confusion_matrix(y_true, y_pred)

    print(f"\n{'=' * 70}")
    print(f"  EVALUATION : {name}")
    print(f"{'=' * 70}")

    # -- Metrics Table --
    print(f"\n    Metric              Score")
    print(f"    {'---':.<20s} {'---':.<10s}")
    print(f"    {'Accuracy':<20s} {acc:.4f}  ({acc:.2%})")
    print(f"    {'Precision':<20s} {prec:.4f}  ({prec:.2%})")
    print(f"    {'Recall':<20s} {rec:.4f}  ({rec:.2%})")
    print(f"    {'F1-Score':<20s} {f1:.4f}  ({f1:.2%})")

    # -- Confusion Matrix --
    tn, fp, fn, tp = cm.ravel()
    print(f"\n    Confusion Matrix:")
    print(f"    {'':>20s}  Predicted NO  Predicted YES")
    print(f"    {'Actual NO':<20s}  {tn:>11,}  {fp:>13,}")
    print(f"    {'Actual YES':<20s}  {fn:>11,}  {tp:>13,}")

    print(f"\n    Interpretation:")
    print(f"      True  Negatives (TN) : {tn:>5,}  (correctly predicted NO churn)")
    print(f"      False Positives (FP) : {fp:>5,}  (predicted churn, but didn't)")
    print(f"      False Negatives (FN) : {fn:>5,}  (predicted no churn, but did)")
    print(f"      True  Positives (TP) : {tp:>5,}  (correctly predicted churn)")

    # -- Full Classification Report --
    print(f"\n    Classification Report:")
    print(classification_report(y_true, y_pred, target_names=["No Churn", "Churn"]))

    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1}


# Evaluate both models
lr_metrics  = evaluate_model("LOGISTIC REGRESSION", y_test, lr_pred)
knn_metrics = evaluate_model("K-NEAREST NEIGHBORS (K=5)", y_test, knn_pred)

# =========================================================================
# 6. SIDE-BY-SIDE COMPARISON
# =========================================================================
print(f"\n{'=' * 70}")
print(f"  MODEL COMPARISON : LOGISTIC REGRESSION  vs  KNN (K=5)")
print(f"{'=' * 70}")

comparison = pd.DataFrame({
    "Logistic Regression": lr_metrics,
    "KNN (K=5)": knn_metrics,
})
comparison["Winner"] = comparison.apply(
    lambda row: "LR" if row["Logistic Regression"] > row["KNN (K=5)"]
    else ("KNN" if row["KNN (K=5)"] > row["Logistic Regression"] else "Tie"),
    axis=1
)

print(f"\n{comparison.to_string()}")

# Overall winner
lr_total  = sum(lr_metrics.values())
knn_total = sum(knn_metrics.values())
if lr_total > knn_total:
    winner = "LOGISTIC REGRESSION"
elif knn_total > lr_total:
    winner = "K-NEAREST NEIGHBORS"
else:
    winner = "TIE"

print(f"\n    Overall Best Model : {winner}")
print(f"\n{'=' * 70}")
print(f"  END OF EVALUATION REPORT")
print(f"{'=' * 70}")


## Overfitting vs. Underfitting

Overfitting occurs when a model learns the training data too well, memorizing the noise and exact details rather than the broad patterns. An overfit model achieves near-perfect accuracy on training data but performs poorly on unseen testing data.

Underfitting occurs when a model is too simple to capture the underlying patterns in the data. An underfit model performs poorly on both the training data and the testing data.

### Analysis of Our Models:

**Logistic Regression (Accuracy: ~80%)**: This model shows a strong balance. An 80% test accuracy indicates it is generalizing well to new data, showing no immediate signs of severe underfitting or overfitting.

**K-Nearest Neighbors (Accuracy: ~76%):** KNN performed worse across all metrics. Because KNN calculates physical distance between data points, it often struggles with high-dimensional encoded data. It is likely underfitting the complex relationships that Logistic Regression was able to capture mathematically.

In [ ]:
"""
=============================================================================
 Customer Churn Prediction - Step 4: Visualization Deliverables
=============================================================================
 Plot 1 : Confusion Matrix Heatmap (Logistic Regression)
 Plot 2 : Side-by-Side Metric Comparison Bar Chart (LR vs KNN)
=============================================================================
"""

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")       # non-interactive backend for saving plots
import seaborn as sns
# =========================================================================
# 1. DATA PREPARATION (same pipeline as Steps 2-3)
# =========================================================================
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df.drop(columns=["customerID"], inplace=True)

for col in [c for c in df.columns if str(df[c].dtype) in ["object", "string", "str"]]:
    df[col] = LabelEncoder().fit_transform(df[col])

numerical_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
df[numerical_cols] = StandardScaler().fit_transform(df[numerical_cols])

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# =========================================================================
# 2. TRAIN MODELS
# =========================================================================
lr_model = LogisticRegression(max_iter=1000, random_state=42, solver="lbfgs")
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

knn_model = KNeighborsClassifier(n_neighbors=5, metric="minkowski", p=2)
knn_model.fit(X_train, y_train)
knn_pred = knn_model.predict(X_test)

# =========================================================================
# 3. COMPUTE METRICS
# =========================================================================
lr_metrics = {
    "Accuracy":  accuracy_score(y_test, lr_pred),
    "Precision": precision_score(y_test, lr_pred),
    "Recall":    recall_score(y_test, lr_pred),
    "F1-Score":  f1_score(y_test, lr_pred),
}
knn_metrics = {
    "Accuracy":  accuracy_score(y_test, knn_pred),
    "Precision": precision_score(y_test, knn_pred),
    "Recall":    recall_score(y_test, knn_pred),
    "F1-Score":  f1_score(y_test, knn_pred),
}
lr_cm = confusion_matrix(y_test, lr_pred)

# =========================================================================
# GLOBAL STYLE
# =========================================================================
sns.set_theme(style="whitegrid", font_scale=1.1)
COLORS = {
    "bg":         "#0F1117",
    "card":       "#1A1D27",
    "text":       "#E8E8EC",
    "accent1":    "#6C63FF",    # vibrant purple
    "accent2":    "#00D4AA",    # teal green
    "grid":       "#2A2D3A",
    "heatmap_cm": "RdPu",      # rich red-purple gradient
}


# =========================================================================
# PLOT 1 : Confusion Matrix Heatmap  (Logistic Regression)
# =========================================================================
fig1, ax1 = plt.subplots(figsize=(8, 6.5))
fig1.patch.set_facecolor(COLORS["bg"])
ax1.set_facecolor(COLORS["card"])

# Heatmap
sns.heatmap(
    lr_cm,
    annot=False,  # Custom annotation below
    cmap=COLORS["heatmap_cm"],
    linewidths=2.5,
    linecolor=COLORS["bg"],
    square=True,
    cbar_kws={"shrink": 0.75, "label": "Count"},
    xticklabels=["Predicted: No", "Predicted: Yes"],
    yticklabels=["Actual: No", "Actual: Yes"],
    ax=ax1,
)

# Title & labels
ax1.set_title(
    "Confusion Matrix - Logistic Regression",
    fontsize=18, fontweight="bold", color=COLORS["text"], pad=20
)
ax1.set_xlabel("Predicted Label", fontsize=13, color=COLORS["text"], labelpad=12)
ax1.set_ylabel("Actual Label", fontsize=13, color=COLORS["text"], labelpad=12)

# Tick styling
ax1.tick_params(colors=COLORS["text"], labelsize=12)
plt.setp(ax1.get_xticklabels(), color=COLORS["text"], fontweight="semibold")
plt.setp(ax1.get_yticklabels(), color=COLORS["text"], fontweight="semibold", rotation=0)

# Colorbar text color
cbar = ax1.collections[0].colorbar
cbar.ax.yaxis.set_tick_params(color=COLORS["text"])
plt.setp(cbar.ax.get_yticklabels(), color=COLORS["text"])
cbar.set_label("Count", color=COLORS["text"], fontsize=12)

# Custom annotations to ensure high visibility contrast
tn, fp, fn, tp = lr_cm.ravel()
cell_specs = [
    (0, 0, tn, "TN", "#FFFFFF", "#E8E8EC"),
    (0, 1, fp, "FP", "#1A1D27", "#2A2D3A"),
    (1, 0, fn, "FN", "#1A1D27", "#2A2D3A"),
    (1, 1, tp, "TP", "#1A1D27", "#2A2D3A"),
]
for i, j, val, name, val_color, name_color in cell_specs:
    ax1.text(j + 0.5, i + 0.42, f"{val}", ha="center", va="center",
             fontsize=28, fontweight="bold", color=val_color)
    ax1.text(j + 0.5, i + 0.72, f"{name} = {val}", ha="center", va="center",
             fontsize=12, fontweight="semibold", fontstyle="italic", color=name_color)

fig1.tight_layout()
fig1.savefig("confusion_matrix_heatmap.png", dpi=200, bbox_inches="tight",
             facecolor=COLORS["bg"], edgecolor="none")
print("[OK] Plot 1 saved: confusion_matrix_heatmap.png")


# =========================================================================
# PLOT 2 : Side-by-Side Metric Comparison Bar Chart
# =========================================================================
metrics = list(lr_metrics.keys())
lr_values  = list(lr_metrics.values())
knn_values = list(knn_metrics.values())

x = np.arange(len(metrics))
bar_width = 0.35

fig2, ax2 = plt.subplots(figsize=(10, 6.5))
fig2.patch.set_facecolor(COLORS["bg"])
ax2.set_facecolor(COLORS["card"])

# Bars with gradient-like effect via edgecolor
bars_lr = ax2.bar(
    x - bar_width/2, lr_values, bar_width,
    label="Logistic Regression",
    color=COLORS["accent1"],
    edgecolor="#8A82FF",
    linewidth=1.5,
    alpha=0.92,
    zorder=3,
)
bars_knn = ax2.bar(
    x + bar_width/2, knn_values, bar_width,
    label="KNN (K=5)",
    color=COLORS["accent2"],
    edgecolor="#33FFD1",
    linewidth=1.5,
    alpha=0.92,
    zorder=3,
)

# Value labels on top of each bar
for bar in bars_lr:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 0.012,
             f"{h:.1%}", ha="center", va="bottom",
             fontsize=11, fontweight="bold", color=COLORS["accent1"])

for bar in bars_knn:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 0.012,
             f"{h:.1%}", ha="center", va="bottom",
             fontsize=11, fontweight="bold", color=COLORS["accent2"])

# Axes styling
ax2.set_xticks(x)
ax2.set_xticklabels(metrics, fontsize=13, fontweight="semibold", color=COLORS["text"])
ax2.set_ylabel("Score", fontsize=14, color=COLORS["text"], labelpad=12)
ax2.set_ylim(0, 1.05)
ax2.set_yticks(np.arange(0, 1.1, 0.1))
ax2.tick_params(axis="y", colors=COLORS["text"], labelsize=11)
ax2.tick_params(axis="x", colors=COLORS["text"])

# Grid
ax2.yaxis.grid(True, color=COLORS["grid"], linestyle="--", alpha=0.5, zorder=0)
ax2.xaxis.grid(False)

# Title
ax2.set_title(
    "Model Comparison: Logistic Regression vs KNN",
    fontsize=18, fontweight="bold", color=COLORS["text"], pad=20
)

# Legend
legend = ax2.legend(
    fontsize=12, loc="upper right",
    framealpha=0.8,
    facecolor=COLORS["card"],
    edgecolor=COLORS["grid"],
)
for text in legend.get_texts():
    text.set_color(COLORS["text"])

# Spine styling
for spine in ax2.spines.values():
    spine.set_color(COLORS["grid"])
    spine.set_linewidth(0.8)

fig2.tight_layout()
fig2.savefig("model_comparison_chart.png", dpi=200, bbox_inches="tight",
             facecolor=COLORS["bg"], edgecolor="none")
print("[OK] Plot 2 saved: model_comparison_chart.png")

print("\n[DONE] Both visualizations saved successfully.")
